# CS336 Spring 2026 - Correct BPE Trainer
----
本节将前面已经完成的组件整合起来:
```text
raw text
    ↓
special-token isolation
    ↓
GPT-style pre-tokenization
    ↓
UTF-8 bytes
    ↓
pretoken → frequency
    ↓
weighted adjacent-pair counts
    ↓
tie-breaking
    ↓
non-overlapping merge
    ↓
vocab + ordered merges
```
> 本节暂时不追求 fast, parallel, memory efficient.
>
> 而是先建立一个足够简单, 完全 deterministic, 能够作为以后优化版本的 reference oracle 的 BPE trainer.

## 1. Reference Trainer 内部应存什么
这一版暂时不在训练内部把 token 表示为 integer ID. 而是直接使用 bytes 表示 token. 

- Trainer 内部表示用 `tuple[bytes, ...]`, 方便观察
- 最终 Assignment 输出仍然是 `vocab: dict[int, bytes]` 和 `merges: list[tuple[bytes, bytes]]`

In [7]:
from __future__ import annotations

from collections import Counter
from typing import TypeAlias

import regex

Token: TypeAlias = bytes
TokenSequence: TypeAlias = tuple[Token, ...]
PretokenCounts: TypeAlias = Counter[TokenSequence]
Pair: TypeAlias = tuple[Token, Token]

## 2. 接回上一节 Pre-tokenization
上一节已经完成了 GPT-style pre-tokenization. 下面继续使用:
```text
ordinary span
    ↓
GPT regex
    ↓
pre-token
```
> 这里有一个必须保持的 invariant: BPE merge 永远不能跨 pre-token boundary.

In [2]:
GPT2_PRETOKEN_PATTERN = (
    r"'(?:[sdmt]|ll|ve|re)"
    r"| ?\p{L}+"
    r"| ?\p{N}+"
    r"| ?[^\s\p{L}\p{N}]+"
    r"|\s+(?!\S)"
    r"|\s+"
)


def pretokenize(text: str) -> list[str]:

    return [match.group(0) for match in regex.finditer(GPT2_PRETOKEN_PATTERN, text)]


print(pretokenize("the cat in the hat"))

['the', ' cat', ' in', ' the', ' hat']


## 3. Special Token 必须先被隔离
上节说到, ordinary pre-tokenization 之前还存在更强的一层 boundary: special-token boundary. 所以训练的顺序必须是:
```text
whole text
    ↓
special-token segmentation
    ↓
ORDINARY / SPECIAL
    ↓
只处理 ORDINARY
    ↓
GPT pre-tokenization
```

In [4]:
def build_special_pattern(special_tokens: list[str]) -> regex.Pattern | None:
    if not special_tokens:
        return None

    ordered = sorted(set(special_tokens), key=lambda token: (-len(token), token))
    alternatives = "|".join(regex.escape(token) for token in ordered)

    return regex.compile(f"({alternatives})")


In [5]:
def split_around_special_tokens(
    text: str, special_tokens: list[str]
) -> list[tuple[bool, str]]:
    pattern = build_special_pattern(special_tokens)

    if pattern is None:
        return [(False, text)] if text else []

    special_set = set(special_tokens)
    result: list[tuple[bool, str]] = []

    for piece in pattern.split(text):
        if not piece:
            continue

        result.append((piece in special_set, piece))

    return result


In [6]:
SPECIAL = "<|endoftext|>"

print(split_around_special_tokens(f"the{SPECIAL}the", [SPECIAL]))

[(False, 'the'), (True, '<|endoftext|>'), (False, 'the')]


## 4. 维护 `pretoken -> frequency`
假设 corpus 中 `the` 出现了 1000000 次. 我们只需保存
```text
(t, h, e)
    ↓
frequency = 1000000
```
因此 trainer 的核心 corpus representation 可以变成:
```text
pre-token token sequence
    ↓
frequency
```
若第 $w$ 个 pre-token 的频率为 $f_w$, pair $p$ 在它当前 token sequence 中出现 $c_w(p)$ 次, 那么:
$$
C(p)=\sum_w f_w c_w(p)
$$
其中 `C(p)` 是整个 corpus 上该 pair 的 weighted frequency.

In [8]:
def count_training_pretokens(text: str, special_tokens: list[str]) -> PretokenCounts:
    counts: PretokenCounts = Counter()

    for is_special, span in split_around_special_tokens(text, special_tokens):
        if is_special:
            continue

        for piece in pretokenize(span):
            byte_tokens = tuple(bytes([byte]) for byte in piece.encode("utf-8"))

            counts[byte_tokens] += 1

    return counts


In [9]:
pretoken_counts = count_training_pretokens(("the cat<|endoftext|>the cat"), [SPECIAL])

for token_sequence, frequency in pretoken_counts.items():
    print(
        [token.decode("utf-8", errors="replace") for token in token_sequence],
        "->",
        frequency,
    )


['t', 'h', 'e'] -> 2
[' ', 'c', 'a', 't'] -> 2
